In [1]:
import pickle
import sys
import copy
import time

import cobra
import sympy
import pandas as pd
import numpy as np

from tqdm import tqdm

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params

No objective coefficients in model. Unclear what should be optimized


In [2]:
from macromolecules.RNA import RNA, mRNA
from macromolecules.protein import Protein
from macromolecules.macromolecule import Macromolecule
from macromolecules.complex import Complex

In [3]:
lp_path = '/data2/hratch/human_me/test_lp/'
with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
    me_model = pickle.load(handle)

In [4]:
# not Metabolites, not proxy (on its own), not mRNA on its own
# HGNC yes


In [5]:
lp_path = '/data2/hratch/human_me/test_lp/'

def add_metabolite(am = [], mu_val = 1e-9):
    
    with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
        me_model = pickle.load(handle)
    
    ra = []
    for m in am: #am:
        r = cobra.Reaction('TEST_' + m.id)
        r.add_metabolites({m: 1})
        ra.append(r)
    
    me_model.add_reactions(ra)
    sln, status, _ = me_model.solve_lp(mu_val = mu_val)
    return sln, status



In [43]:
def flatten_list(list_):
    return [item for sublist in list_ for item in sublist]


test = sorted(set(flatten_list([[m.id for m in r.metabolites] for r in err])))
am = [m for m in me_model.metabolites if '_unfolded_protein_c' in m.id and m.id in test]
sln, status = add_metabolite(am = am)

Getting MINOS parameters...
Done in 219.833 seconds with status 0


['HGNC:10311_TRANSCRIPTION_ELONGATION',
 'HGNC:10311_lariats_DEGRADATIONn_0',
 'HGNC:10311_TRANSCRIPTION_PROCESSING',
 'HGNC:10311_mRNA_EXPORTtn',
 'HGNC:10311_DECAPPING_mRNA_DEGRADATIONc',
 'HGNC:10311_TRANSLATION_ELONGATIONc',
 'HGNC:10311_CYTOSOLIC_PROTEIN_FOLDING',
 'HGNC:10311_folded_protein_c_POLYUBIQUITINATIONc',
 'HGNC:10311_folded_protein_c_DEUBIQUITINATIONc',
 'HGNC:10311_folded_protein_c_PROTEASOMAL_DEGRADATIONc',
 'HGNC:10311_IMPORTtn']

In [156]:
test = sorted(set(flatten_list([[m.id for m in r.metabolites] for r in me_model.reactions if r.subsystem == 'Ribosome Biogenesis' and isinstance(r, func.ME_Reaction) and 'translation' in r.type])))
example = [m.id for m in me_model.metabolites if 'unfolded_protein_c' in m.id and m.id not in err and m.id in test]
example = me_model.metabolites.get_by_id(example[-9])
fail = me_model.metabolites.get_by_id(err[0])

example = [r.id for r in me_model.reactions if example.id.split('_')[0] in r.id]
fail = [r.id for r in me_model.reactions if fail.id.split('_')[0] in r.id]

In [157]:
for r_id in fail:
    r = me_model.reactions.get_by_id(r_id)
    print(r.id + ': ' + str(sln[me_model.reactions.index(r.id)]))

HGNC:10420_TRANSCRIPTION_ELONGATION: 1.1810288850956215e-58
HGNC:10420_lariats_DEGRADATIONn_0: 0.0
HGNC:10420_TRANSCRIPTION_PROCESSING: 0.0
HGNC:10420_mRNA_EXPORTtn: 0.0
HGNC:10420_DECAPPING_mRNA_DEGRADATIONc: 0.0
HGNC:10420_TRANSLATION_ELONGATIONc: 0.0
HGNC:10420_CYTOSOLIC_PROTEIN_FOLDING: 2.522462186588349e-17
HGNC:10420_folded_protein_c_POLYUBIQUITINATIONc: 0.0
HGNC:10420_folded_protein_c_DEUBIQUITINATIONc: 0.0
HGNC:10420_folded_protein_c_PROTEASOMAL_DEGRADATIONc: 0.0
HGNC:10420_IMPORTtn: 2.522462186588349e-17


In [ ]:
[m.id for m in ]

In [153]:
for r_id in example:
    r = me_model.reactions.get_by_id(r_id)
    print(r.id + ': ' + str(sln[me_model.reactions.index(r.id)]))

HGNC:10313_TRANSCRIPTION_ELONGATION: 1.0150427697272839e-22
HGNC:10313_lariats_DEGRADATIONn_0: 1.0150427697272839e-22
HGNC:10313_TRANSCRIPTION_PROCESSING: 1.0150427697272839e-22
HGNC:10313_mRNA_EXPORTtn: 1.0150427697272839e-22
HGNC:10313_DECAPPING_mRNA_DEGRADATIONc: 5.075213812026491e-23
HGNC:10313_TRANSLATION_ELONGATIONc: 2.522462186588349e-17
HGNC:10313_CYTOSOLIC_PROTEIN_FOLDING: 2.522462186588349e-17
HGNC:10313_folded_protein_c_POLYUBIQUITINATIONc: 0.0
HGNC:10313_folded_protein_c_DEUBIQUITINATIONc: 0.0
HGNC:10313_folded_protein_c_PROTEASOMAL_DEGRADATIONc: 0.0
HGNC:10313_IMPORTtn: 2.522462186588349e-17


In [154]:
import cobra
import warnings

import pandas as pd
import numpy as np

from Bio.Seq import Seq
from Bio import SeqIO

import sys
sys.path.insert(1, '../../scripts/') # comment out in python script
from utils.load_environmental_variables import raw_data_path
from utils import machinery as mach
from utils import parameters as params
from utils import metabolites as metab
from utils import functions as func
from utils import utils_2

from macromolecules.RNA import rRNA, RNA_fragment
from macromolecules.protein import Protein
from macromolecules.complex import Complex, add_biomass_change

import expression.build_mrna_expression_reactions as build_mrna
from expression.protein_expression import cytosolic_translation as c_trln
from expression.protein_expression import build_protein_expression_reactions as build_protein
from expression.protein_expression import ubiquitin

fail_id = 'HGNC:10420'
example_id = 'HGNC:10311'

psim_rib = params.psim_me.copy()
def format_location(x):
    return ['n', 'c']
psim_rib.LOCATION = psim_rib.LOCATION.apply(lambda x: format_location(x))
compress_mrna = False
ub_args = ubiquitin.express_ubiquitin(compress_mrna)

gene_info = utils_2.generate_geneinfo_object(hgnc_id = fail_id, psim = psim_rib, 
            machinery_list = list(), metabolic_model = cobra.Model())
gene_info.final_locations = {'c': 'Cytosolic Tranport', 'n': 'Cytosolic Tranport'}
mrna_expression_reactions, mrna_transcript_c, mrna_deg_proxy = build_mrna.get_mrna_expression_reactions(gene_info, compress_mrna = compress_mrna)
protein_expression_reactions, protein_metabolites = build_protein.get_protein_expression_reactions(gene_info, mrna_transcript_c, mrna_deg_proxy, ub_args)
protein_expression_reactions = protein_expression_reactions[:-3] # no nuclear degradation
fail_ptr = gene_info.ptr

fail_reactions_n = protein_expression_reactions + mrna_expression_reactions

gene_info = utils_2.generate_geneinfo_object(hgnc_id = example_id, psim = psim_rib, 
            machinery_list = list(), metabolic_model = cobra.Model())
gene_info.final_locations = {'c': 'Cytosolic Tranport', 'n': 'Cytosolic Tranport'}
mrna_expression_reactions, mrna_transcript_c, mrna_deg_proxy = build_mrna.get_mrna_expression_reactions(gene_info, compress_mrna = compress_mrna)
protein_expression_reactions, protein_metabolites = build_protein.get_protein_expression_reactions(gene_info, mrna_transcript_c, mrna_deg_proxy, ub_args)
protein_expression_reactions = protein_expression_reactions[:-3] # no nuclear degradation

example_reactions_n = protein_expression_reactions + mrna_expression_reactions

No errors raised
No errors raised


In [155]:
gene_info.ptr

65012.96903430903